<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px 30px; border-radius: 12px; margin-bottom: 10px;">
    <h1 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:2.2em; margin:0 0 8px 0;">
        🎓 Introdução ao Aprendizado de Máquina
    </h1>
    <h2 style="color:#a8d8ea; font-family:'Segoe UI', sans-serif; font-size:1.3em; margin:0 0 6px 0; font-weight:400;">
        Aula 05 — K-Nearest Neighbors (KNN)
    </h2>
    <h3 style="color:#e2b96f; font-family:'Segoe UI', sans-serif; font-size:1.05em; margin:0 0 12px 0; font-weight:500;">
        🔬 Prática — Nosso Primeiro Modelo no Titanic
    </h3>
    <p style="color:#ccc; font-family:'Segoe UI', sans-serif; font-size:0.9em; margin:0;">
        Prof. Felipe Amaral
    </p>
</div>

<div style="display:flex; gap:10px; margin-top:10px; flex-wrap:wrap;">
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">📚 FIAP</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🐍 Python 3</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🚢 Dataset Titanic</span>
    <span style="background:#0f3460; color:#a8d8ea; padding:5px 14px; border-radius:20px; font-size:0.85em;">🤖 Primeiro Modelo!</span></div>


## Chegou a hora

Nas últimas aulas você explorou, limpou e preparou os dados do Titanic.
Hoje esse trabalho tem recompensa: **vamos treinar o primeiro modelo de Machine Learning do curso**.

O algoritmo escolhido é o **K-Nearest Neighbors (KNN)** — o mais intuitivo de todos.
Antes de qualquer matemática avançada, ele faz algo que você faz intuitivamente todos os dias:

> *"Para classificar algo desconhecido, olhe para os exemplos mais parecidos que você já viu."*

---

## Roteiro de hoje

| Parte | Tema |
|-------|------|
| **Config** | Recarregando e preparando o Titanic |
| **1** | Intuição do KNN — sem código, só raciocínio |
| **2** | Distância: a matemática por trás do "parecido" |
| **3** | Nosso primeiro modelo — `fit` e `predict` |
| **4** | O efeito da escala — normalização importa |
| **5** | Escolhendo o K ideal |
| **6** | Avaliando o modelo: métricas completas |

<div style="background:#d4edda; border-left:5px solid #155724; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#155724;">✅ </strong><span style="color:#155724;">Este é o notebook mais importante do curso até agora: você vai usar <code>modelo.fit()</code> e <code>modelo.predict()</code> pela primeira vez. Tudo que preparamos nas aulas anteriores foi para chegar aqui.</span></div>

---

## Configuração — Execute antes de começar


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.figsize":  (10, 5),
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.titlesize":     13,
    "axes.labelsize":     11,
})
sns.set_theme(style="whitegrid", palette="muted")

# ── Reproduzindo todo o pré-processamento das aulas anteriores ────────────────
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df_raw = sns.load_dataset("titanic")
df = df_raw.copy()

# Limpeza (Aula 03)
df["age"]      = df["age"].fillna(df["age"].median())
df["embarked"] = df["embarked"].fillna(df["embarked"].mode()[0])
df = df.drop(columns=["deck"]).drop_duplicates().reset_index(drop=True)

# Feature Engineering (Aula 04)
df["tamanho_familia"]   = df["sibsp"] + df["parch"] + 1
df["sozinho"]           = (df["tamanho_familia"] == 1).astype(int)
df["faixa_etaria"]      = pd.cut(df["age"], bins=[0,12,18,60,100],
                                  labels=["Criança","Adolescente","Adulto","Idoso"])
df["faixa_etaria_enc"]  = df["faixa_etaria"].map(
                            {"Criança":0,"Adolescente":1,"Adulto":2,"Idoso":3})
df["titulo"]            = df["name"].str.extract(r",\s([A-Za-z]+)\.")
df["titulo"]            = df["titulo"].map(
    lambda t: t if t in ["Mr","Miss","Mrs","Master"] else "Raro")
df["tarifa_por_pessoa"] = (df["fare"] / df["tamanho_familia"].clip(lower=1)).round(2)
df["sex_enc"]           = (df["sex"] == "female").astype(int)
df["pclass_enc"]        = df["pclass"].map({1:2, 2:1, 3:0})

embarked_ohe = pd.get_dummies(df["embarked"], prefix="embarked", drop_first=True)
titulo_ohe   = pd.get_dummies(df["titulo"],   prefix="titulo",   drop_first=True)
df = pd.concat([df, embarked_ohe, titulo_ohe], axis=1)

# Split + Normalização (Aula 04)
FEATURES = ["pclass_enc","sex_enc","age","tamanho_familia","sozinho",
            "faixa_etaria_enc","tarifa_por_pessoa",
            "embarked_q","embarked_s",
            "titulo_Master","titulo_Miss","titulo_Mr","titulo_Mrs","titulo_Raro"]

# Garantindo que todas as colunas existem
FEATURES = [f for f in FEATURES if f in df.columns]

X = df[FEATURES].copy()
y = df["survived"].copy()

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_treino_sc = scaler.fit_transform(X_treino)
X_teste_sc  = scaler.transform(X_teste)

# Versões sem normalização (usaremos na Parte 4)
X_treino_raw = X_treino.values
X_teste_raw  = X_teste.values

print("✅ Dataset pronto! Retomando de onde paramos.")
print(f"   Treino:  {X_treino_sc.shape[0]} amostras  |  {X_treino_sc.shape[1]} features")
print(f"   Teste:   {X_teste_sc.shape[0]}  amostras  |  {X_teste_sc.shape[1]} features")
print(f"   Features: {FEATURES}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 1</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Intuição do KNN — Classificar pelo que é Parecido</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Diga-me com quem andas e te direi quem és."</p></div><</div></div>


### A ideia central

O KNN faz uma coisa simples e poderosa:

> Para classificar um novo ponto, encontre os **K exemplos mais parecidos** no conjunto
> de treino e faça uma **votação**: a classe mais votada é a previsão.

Imagine que você recebe um passageiro novo do Titanic e precisa prever se sobreviveu.
O KNN procura os K passageiros mais parecidos (por idade, classe, gênero...) e pergunta:
*"Dos K mais parecidos, quantos sobreviveram?"*

---

### Três características importantes

| Característica | Significado | Consequência prática |
|----------------|-------------|---------------------|
| **Lazy Learning** | Não há treinamento real — o modelo guarda os dados na memória | `fit()` é instantâneo; `predict()` é lento em datasets grandes |
| **Não-paramétrico** | Não assume nenhuma forma para a fronteira de decisão | Flexível, mas pode overfittar com K pequeno |
| **Baseado em distância** | "Parecido" = distância pequena | **Normalização é obrigatória** |


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 1 — Antes de qualquer código, olhe para a tabela abaixo e responda: se K=3, qual classe o KNN prevê para o Passageiro Novo? E se K=5? Complete a tabela de respostas abaixo.</span></div>

### Exercício manual — Votação KNN

O **Passageiro Novo** tem: Idade=28, Classe=1ª, Gênero=Feminino.

Os 5 vizinhos mais próximos encontrados são:

| Vizinho | Distância | Idade | Classe | Gênero | Sobreviveu? |
|---------|-----------|-------|--------|--------|-------------|
| V1 | 0.8 | 26 | 1ª | Feminino | ✅ Sim |
| V2 | 1.1 | 30 | 1ª | Feminino | ✅ Sim |
| V3 | 1.4 | 25 | 2ª | Feminino | ✅ Sim |
| V4 | 1.9 | 32 | 1ª | Masculino | ❌ Não |
| V5 | 2.3 | 27 | 3ª | Feminino | ✅ Sim |

*✏️ Com K=3: o modelo analisa V1, V2 e V3. Votos: ___ Sim, ___ Não. Previsão: ___*

*✏️ Com K=5: o modelo analisa todos. Votos: ___ Sim, ___ Não. Previsão: ___*

*✏️ Esse passageiro novo realmente sobreviveu? Com base nos dados do Titanic, faz sentido?*


In [ ]:
# ── GABARITO DA MISSÃO 1 (descomente para ver) ───────────────────────────────
# print("Gabarito:")
# print()
# print("K=3: analisa V1, V2, V3")
# print("  Votos: 3 Sim, 0 Não → Previsão: SOBREVIVEU")
# print()
# print("K=5: analisa V1, V2, V3, V4, V5")
# print("  Votos: 4 Sim, 1 Não → Previsão: SOBREVIVEU")
# print()
# print("Faz sentido? SIM!")
# print("Mulher, 1ª classe, jovem → taxa de sobrevivência histórica ~97%")
# print("O KNN captura exatamente esse padrão nos dados.")


---

### Visualizando a Fronteira de Decisão

Vamos criar um exemplo 2D simples (apenas 2 features) para ver graficamente
como o KNN divide o espaço em regiões de decisão.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification

# Dataset 2D simples para visualização
np.random.seed(42)
X_vis, y_vis = make_classification(
    n_samples=120, n_features=2, n_redundant=0,
    n_informative=2, n_clusters_per_class=1,
    class_sep=1.0, random_state=42
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("KNN — Fronteira de Decisão para K=1, K=5 e K=15",
             fontsize=13, fontweight="bold")

Ks = [1, 5, 15]
h  = 0.04

for ax, k in zip(axes, Ks):
    # Treinar KNN com K vizinhos
    knn_vis = KNeighborsClassifier(n_neighbors=k)
    knn_vis.fit(X_vis, y_vis)

    # Criar grade de pontos para colorir o fundo (fronteira de decisão)
    x_min, x_max = X_vis[:,0].min()-0.5, X_vis[:,0].max()+0.5
    y_min, y_max = X_vis[:,1].min()-0.5, X_vis[:,1].max()+0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                          np.arange(y_min, y_max, h))
    Z = knn_vis.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Colorindo as regiões de decisão
    ax.contourf(xx, yy, Z, alpha=0.25, cmap="RdBu")
    ax.contour(xx, yy, Z, colors="gray", linewidths=0.5, alpha=0.5)

    # Plotando os pontos de treino
    scatter = ax.scatter(X_vis[:,0], X_vis[:,1], c=y_vis,
                          cmap="RdBu", edgecolors="white",
                          linewidth=0.6, s=55, zorder=5)

    score = knn_vis.score(X_vis, y_vis)
    ax.set_title(f"K = {k}   (acurácia treino: {score:.0%})",
                 fontweight="bold")
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")

plt.tight_layout()
plt.savefig("aula05_fronteiras_knn.png", dpi=110, bbox_inches="tight")
plt.show()

print("Observe:")
print("  K=1 → fronteira muito irregular, decora o treino (overfitting)")
print("  K=5 → fronteira suave e razoável")
print("  K=15 → fronteira muito simples, pode perder padrões (underfitting)")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 2</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Distância — A Matemática Por Trás de 'Parecido'</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"O KNN mede o quanto dois passageiros se parecem com distância matemática."</p></div></div></div>


### Como medir "parecido"?

O KNN precisa de uma forma objetiva de dizer o quão parecidos dois exemplos são.
A resposta é a **distância** entre eles no espaço de features.

### Distância Euclidiana — a mais comum

É a distância em linha reta, como a régua mede no papel:

```
           √
d(A, B) =   (A₁−B₁)² + (A₂−B₂)² + ... + (Aₙ−Bₙ)²
```

Com 2 features (ex: idade e tarifa), é o Teorema de Pitágoras:

```
         Passageiro A: Idade=25, Tarifa=50
         Passageiro B: Idade=30, Tarifa=80

         d = √((25−30)² + (50−80)²)
           = √(25 + 900)
           = √925
           ≈ 30,4
```

### Outras métricas de distância

| Métrica | Fórmula | Quando usar |
|---------|---------|-------------|
| **Euclidiana** | √Σ(pᵢ−qᵢ)² | Padrão — funciona bem na maioria dos casos |
| **Manhattan** | Σ\|pᵢ−qᵢ\| | Dados com muitas dimensões; menos sensível a outliers |
| **Minkowski** | (Σ\|pᵢ−qᵢ\|ᵖ)^(1/p) | Generalização que inclui Euclidiana (p=2) e Manhattan (p=1) |
| **Cosseno** | 1 − cos(θ) | Textos e recomendações — mede ângulo, não magnitude |


In [ ]:
# Calculando distância euclidiana manualmente — passo a passo
import numpy as np

# Dois passageiros do Titanic (valores já normalizados — veremos por quê depois)
p_ana   = np.array([2.0, 1.0, 0.3, 1.0])   # pclass_enc, sex_enc, age_norm, tamanho_familia
p_bruno = np.array([0.0, 0.0, 1.2, 3.0])

print("CÁLCULO MANUAL — Distância Euclidiana")
print("=" * 50)
print(f"Ana:   {p_ana}")
print(f"Bruno: {p_bruno}")
print()

# Passo 1: diferença
diff = p_ana - p_bruno
print(f"Passo 1 — Diferenças:         {diff}")

# Passo 2: quadrado
diff2 = diff ** 2
print(f"Passo 2 — Quadrados:          {diff2}")

# Passo 3: soma
soma = diff2.sum()
print(f"Passo 3 — Soma dos quadrados: {soma:.4f}")

# Passo 4: raiz
dist = np.sqrt(soma)
print(f"Passo 4 — Raiz quadrada:      {dist:.4f}")

# Verificando com numpy
dist_numpy = np.linalg.norm(p_ana - p_bruno)
print(f"
✅ Verificação (np.linalg.norm): {dist_numpy:.4f} — idêntico!")


In [ ]:
# Comparação visual: Euclidiana vs Manhattan
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Euclidiana vs Manhattan — Dois Caminhos para Medir Distância",
             fontweight="bold")

A = np.array([1, 1])
B = np.array([5, 4])

for ax, titulo, cor_caminho in zip(axes,
    ["Distância Euclidiana
(linha reta)", "Distância Manhattan
(blocos de cidade)"],
    ["#0f3460", "#e94560"]):

    # Pontos
    ax.scatter(*A, s=150, color="#0f3460", zorder=5, label="Ponto A")
    ax.scatter(*B, s=150, color="#e94560", zorder=5, label="Ponto B")
    ax.annotate("A", A, textcoords="offset points", xytext=(8,5), fontsize=12, fontweight="bold")
    ax.annotate("B", B, textcoords="offset points", xytext=(8,5), fontsize=12, fontweight="bold")

    if "Euclidiana" in titulo:
        ax.plot([A[0], B[0]], [A[1], B[1]], color=cor_caminho, linewidth=2.5,
                label=f"d = {np.linalg.norm(B-A):.2f}")
    else:
        ax.plot([A[0], B[0], B[0]], [A[1], A[1], B[1]],
                color=cor_caminho, linewidth=2.5,
                label=f"d = {abs(B[0]-A[0]) + abs(B[1]-A[1]):.2f}")

    ax.set_xlim(0, 7); ax.set_ylim(0, 6)
    ax.set_title(titulo, fontweight="bold")
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

print(f"Euclidiana: {np.linalg.norm(B-A):.3f}  (linha reta)")
print(f"Manhattan:  {abs(B[0]-A[0]) + abs(B[1]-A[1]):.3f}  (horizontal + vertical)")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 2 — Calcule manualmente a distância Euclidiana entre os Passageiros C e D abaixo. Depois use <code>np.linalg.norm()</code> para verificar. Qual está mais perto do Passageiro A?</span></div>

In [ ]:
# ✏️ Passageiros para calcular (features já normalizadas)
p_ref = np.array([1.5,  1.0,  0.5,  1.0])   # Passageiro Referência
p_C   = np.array([1.2,  1.0,  0.3,  1.0])   # Passageiro C
p_D   = np.array([0.0,  0.0,  1.8,  4.0])   # Passageiro D

# ✏️ Calcule manualmente aqui:
# dist_C = ???
# dist_D = ???

# Verifique com numpy:
# print(f"Distância Ref → C: {np.linalg.norm(p_ref - p_C):.4f}")
# print(f"Distância Ref → D: {np.linalg.norm(p_ref - p_D):.4f}")
# print(f"O Passageiro ??? está mais próximo da Referência.")
print("Complete o cálculo acima!")


In [ ]:
# ── GABARITO DA MISSÃO 2 (descomente para ver) ───────────────────────────────
# p_ref = np.array([1.5,  1.0,  0.5,  1.0])
# p_C   = np.array([1.2,  1.0,  0.3,  1.0])
# p_D   = np.array([0.0,  0.0,  1.8,  4.0])
#
# dist_C = np.linalg.norm(p_ref - p_C)
# dist_D = np.linalg.norm(p_ref - p_D)
#
# print(f"Distância Ref → C: {dist_C:.4f}")
# print(f"Distância Ref → D: {dist_D:.4f}")
# print(f"Passageiro C está mais próximo (dist={dist_C:.4f} < {dist_D:.4f})")
# print()
# print("C e Referência têm idade e gênero muito parecidos — faz sentido!")


---

### Implementando o KNN do Zero (sem sklearn)

Antes de usar a biblioteca, vamos entender exatamente o que acontece por dentro.


In [ ]:
def knn_do_zero(X_treino, y_treino, x_novo, k=3):
    # KNN implementado do zero - para entender o algoritmo por dentro.
    # Passos:
    # 1. Calcular a distância de x_novo para TODOS os pontos de treino
    # 2. Ordenar do mais próximo ao mais distante
    # 3. Pegar os K primeiros (vizinhos mais próximos)
    # 4. Fazer votação: classe com mais votos é a previsão
    # Passo 1: calcular todas as distâncias
    distancias = []
    for i, x_treino in enumerate(X_treino):
        dist = np.linalg.norm(x_novo - x_treino)
        distancias.append((dist, y_treino[i], i))

    # Passo 2: ordenar pelo mais próximo
    distancias.sort(key=lambda x: x[0])

    # Passo 3: pegar os K vizinhos
    k_vizinhos = distancias[:k]

    # Passo 4: votação
    votos = {}
    for dist, classe, idx in k_vizinhos:
        votos[classe] = votos.get(classe, 0) + 1

    # Classe com mais votos
    previsao = max(votos, key=votos.get)

    return previsao, k_vizinhos, votos


# ── Testando com os dados do Titanic ──────────────────────────────────────────
# Usando apenas 2 features para facilitar a compreensão
X_2feat = X_treino_sc[:, :2]     # pclass_enc e sex_enc normalizados
y_arr   = y_treino.values

# Novo passageiro: mulher de 1ª classe (features normalizadas)
x_novo_norm = X_teste_sc[0, :2]
label_real  = y_teste.values[0]

previsao, vizinhos, votos = knn_do_zero(X_2feat, y_arr, x_novo_norm, k=5)

print("KNN DO ZERO — Previsão para um novo passageiro")
print("=" * 52)
print(f"Features do novo passageiro (normalizadas): {x_novo_norm.round(3)}")
print(f"
5 vizinhos mais próximos encontrados:")
print(f"  {'Vizinho':>8} {'Distância':>10} {'Sobreviveu':>12} {'Votos acum.'}")
print("  " + "-"*46)
for i, (dist, classe, idx) in enumerate(vizinhos, 1):
    print(f"  {i:>8} {dist:>10.4f} {'✅ Sim' if classe==1 else '❌ Não':>12}")

print(f"
Contagem de votos: {votos}")
print(f"Previsão do modelo: {'Sobreviveu ✅' if previsao==1 else 'Não sobreviveu ❌'}")
print(f"Valor real:         {'Sobreviveu ✅' if label_real==1 else 'Não sobreviveu ❌'}")
print(f"
O modelo {'ACERTOU ✅' if previsao == label_real else 'ERROU ❌'}!")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 3</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Nosso Primeiro Modelo — fit e predict</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Três linhas de código. Uma aula inteira de preparação. Vale cada segundo."</p></div></div></div>


### O momento que preparamos nas últimas aulas

Lembra de tudo que fizemos?

- **Aula 03** → exploramos, limpamos, visualizamos  
- **Aula 04** → criamos features, normalizamos, codificamos, dividimos treino/teste  
- **Aula 05** → entendemos a intuição e a matemática do KNN  

Agora o modelo:


In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# ─────────────────────────────────────────────────────────────────────────────
#  AS TRÊS LINHAS MAIS IMPORTANTES QUE VOCÊ VAI ESCREVER NO CURSO
# ─────────────────────────────────────────────────────────────────────────────

# 1. Criar o modelo
modelo_knn = KNeighborsClassifier(n_neighbors=5)

# 2. Treinar — o modelo aprende com os dados de treino
modelo_knn.fit(X_treino_sc, y_treino)

# 3. Prever — o modelo prevê para dados que nunca viu
y_previsto = modelo_knn.predict(X_teste_sc)

# ─────────────────────────────────────────────────────────────────────────────

print("🎉 Primeiro modelo treinado!")
print()
print("Comparando previsões vs valores reais (primeiros 15 passageiros do teste):")
print()
print(f"  {'#':>3}  {'Real':>10}  {'Previsto':>10}  {'Resultado':>12}")
print("  " + "-"*42)
for i in range(15):
    real     = y_teste.values[i]
    previsto = y_previsto[i]
    real_str = "Sobreviveu" if real == 1 else "Não Sobrev."
    prev_str = "Sobreviveu" if previsto == 1 else "Não Sobrev."
    ok       = "✅ Acerto" if real == previsto else "❌ Erro"
    print(f"  {i+1:>3}  {real_str:>10}  {prev_str:>10}  {ok:>12}")

acertos = (y_previsto == y_teste.values).sum()
total   = len(y_teste)
print(f"
Nesses 15: {sum(y_previsto[:15] == y_teste.values[:15])} acertos de 15")
print(f"No teste completo ({total} passageiros): {acertos} acertos → {acertos/total:.1%}")


<div style="background:#e8d5f5; border-left:5px solid #5b2c8d; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#5b2c8d;">💡 </strong><span style="color:#5b2c8d;">Note que <code>fit()</code> com KNN é instantâneo — o modelo apenas memoriza os dados. O trabalho real acontece no <code>predict()</code>, quando ele calcula distâncias para cada nova previsão. Isso é o que chamamos de <strong>Lazy Learning</strong>.</span></div>

In [ ]:
# Visualizando acertos e erros no conjunto de teste
fig, ax = plt.subplots(figsize=(10, 4))

cores = ["#e94560" if real != prev else "#0f3460"
         for real, prev in zip(y_teste.values, y_previsto)]
rotulos = ["Erro" if real != prev else "Acerto"
           for real, prev in zip(y_teste.values, y_previsto)]

# Scatter: cada ponto é um passageiro do teste
x_pos = range(len(y_teste))
ax.scatter(x_pos, y_teste.values, c=cores, s=30, alpha=0.7, zorder=5)

n_erros  = sum(r != p for r, p in zip(y_teste.values, y_previsto))
n_acertos = len(y_teste) - n_erros

ax.set_xlabel("Índice do Passageiro no Teste")
ax.set_ylabel("Sobreviveu (1=Sim, 0=Não)")
ax.set_title(f"Acertos e Erros no Conjunto de Teste  "
             f"(K=5 | Acertos: {n_acertos} | Erros: {n_erros})",
             fontweight="bold")

patch_acerto = mpatches.Patch(color="#0f3460", label=f"Acerto ({n_acertos})")
patch_erro   = mpatches.Patch(color="#e94560", label=f"Erro ({n_erros})")
ax.legend(handles=[patch_acerto, patch_erro], loc="upper right")

plt.tight_layout(); plt.show()


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 4</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">O Efeito da Escala — Por que Normalização é Obrigatória</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Sem normalização, o KNN ignora variáveis pequenas sem perceber."</p></div></div></div>


### O problema

O KNN mede distâncias. Se uma feature tem valores de 0 a 500 e outra de 0 a 1,
a primeira domina completamente o cálculo — a segunda é praticamente ignorada.

Veja as escalas das nossas features:


In [ ]:
# Visualizando o problema de escala nas features do Titanic
print("Escalas das features ANTES da normalização:")
print("=" * 52)
print(f"  {'Feature':<22} {'Mínimo':>7} {'Máximo':>9} {'Média':>9}")
print("  " + "-"*50)
for col in X_treino.columns:
    mn = X_treino[col].min()
    mx = X_treino[col].max()
    me = X_treino[col].mean()
    print(f"  {col:<22} {mn:>7.2f} {mx:>9.2f} {me:>9.2f}")

print()
print("⚠️  'tarifa_por_pessoa' pode chegar a ~256!")
print("    'sozinho' vai apenas de 0 a 1.")
print("    Sem normalização, a tarifa domina a distância.")


In [ ]:
# Comparação direta: KNN COM vs SEM normalização

# Sem normalização (dados brutos)
knn_sem_norm = KNeighborsClassifier(n_neighbors=5)
knn_sem_norm.fit(X_treino_raw, y_treino)
y_pred_sem = knn_sem_norm.predict(X_teste_raw)
acc_sem = (y_pred_sem == y_teste.values).mean()

# Com normalização (StandardScaler)
knn_com_norm = KNeighborsClassifier(n_neighbors=5)
knn_com_norm.fit(X_treino_sc, y_treino)
y_pred_com = knn_com_norm.predict(X_teste_sc)
acc_com = (y_pred_com == y_teste.values).mean()

print("COMPARAÇÃO: Com vs Sem Normalização (K=5)")
print("=" * 42)
print(f"  Sem normalização:  acurácia = {acc_sem:.4f}  ({acc_sem:.1%})")
print(f"  Com normalização:  acurácia = {acc_com:.4f}  ({acc_com:.1%})")
print(f"
  Ganho de normalizar: +{(acc_com - acc_sem)*100:.1f} pontos percentuais")

# Gráfico comparativo
fig, ax = plt.subplots(figsize=(8, 4))
barras = ax.bar(["Sem Normalização", "Com Normalização (Z-score)"],
                [acc_sem * 100, acc_com * 100],
                color=["#e94560", "#0f3460"],
                edgecolor="white", width=0.5)
for b, v in zip(barras, [acc_sem*100, acc_com*100]):
    ax.text(b.get_x() + b.get_width()/2, v + 0.3,
            f"{v:.1f}%", ha="center", fontsize=13, fontweight="bold")
ax.set_ylabel("Acurácia (%)")
ax.set_ylim(70, 95)
ax.set_title("Impacto da Normalização no KNN (K=5)", fontweight="bold")
plt.tight_layout(); plt.show()


<div style="background:#f8d7da; border-left:5px solid #721c24; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#721c24;">🚨 </strong><span style="color:#721c24;">Este gráfico é a demonstração mais importante desta aula: a mesma implementação do KNN, com os mesmos dados, produz resultados diferentes dependendo da normalização. <strong>Para o KNN, normalizar não é opcional — é parte do algoritmo.</strong></span></div>

In [ ]:
# Por que isso acontece? Demonstração com 2 features extremas
print("DEMONSTRAÇÃO — Distância com escalas diferentes")
print("=" * 55)

# Dois passageiros similares (mesmo perfil de sobrevivência)
# Features: [sozinho (0-1), tarifa_por_pessoa (0-256)]
A = np.array([0, 50.0])    # acompanhado, tarifa média
B = np.array([1, 52.0])    # sozinho,     tarifa similar
C = np.array([0, 200.0])   # acompanhado, tarifa muito alta

d_AB = np.linalg.norm(A - B)
d_AC = np.linalg.norm(A - C)

print(f"Passageiro A: sozinho={A[0]}, tarifa=£{A[1]:.0f}")
print(f"Passageiro B: sozinho={B[0]}, tarifa=£{B[1]:.0f}")
print(f"Passageiro C: sozinho={C[0]}, tarifa=£{C[1]:.0f}")
print()
print(f"Distância A→B (sem norm): {d_AB:.2f}")
print(f"Distância A→C (sem norm): {d_AC:.2f}")
print()
print("O modelo diz que A é mais parecido com C (distância menor)!")
print("Mas B tem o mesmo status familiar que A — a tarifa domina tudo.")
print()

# Após normalização
A_n = (A - np.array([0.3, 30.0])) / np.array([0.45, 40.0])
B_n = (B - np.array([0.3, 30.0])) / np.array([0.45, 40.0])
C_n = (C - np.array([0.3, 30.0])) / np.array([0.45, 40.0])

d_AB_n = np.linalg.norm(A_n - B_n)
d_AC_n = np.linalg.norm(A_n - C_n)

print(f"Após normalização:")
print(f"  Distância A→B: {d_AB_n:.4f}")
print(f"  Distância A→C: {d_AC_n:.4f}")
print(f"  Agora A e B são reconhecidos como mais parecidos. ✅")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 5</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Escolhendo o K Ideal</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"K pequeno decora, K grande simplifica demais. Onde está o equilíbrio?"</p></div></div></div>


### O dilema do K

| K | Fronteira | Risco | Analogia |
|---|-----------|-------|----------|
| **K=1** | Muito irregular | Overfitting — decora o treino | Copiar a resposta do vizinho mais próximo |
| **K muito grande** | Muito suave | Underfitting — ignora padrões locais | Votar com todos — a maioria sempre ganha |
| **K ideal** | Suave e precisa | Bom equilíbrio | Consultar um grupo razoável de especialistas |

### Regra prática para começar

```
K_inicial = √(número de amostras de treino)
```

No nosso caso: √712 ≈ 26 — mas sempre teste outros valores!


In [ ]:
# Calculando K inicial pela regra da raiz quadrada
n_treino = len(X_treino_sc)
k_raiz   = int(np.sqrt(n_treino))

print(f"Amostras de treino: {n_treino}")
print(f"K inicial (√n):     {k_raiz}")
print(f"Usar ímpar?         {k_raiz if k_raiz % 2 == 1 else k_raiz + 1} (preferível em classificação binária)")
print()
print("Por que ímpar? Para evitar empates na votação.")
print(f"  K={k_raiz} (par):    pode empatar — ex: 13 Sim vs 13 Não")
print(f"  K={k_raiz+1} (ímpar): desempate garantido — ex: 13 vs 14")


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 3 — Complete o loop abaixo para testar diferentes valores de K (de 1 a 30) e plotar a acurácia. Identifique: (a) onde ocorre overfitting, (b) onde ocorre underfitting, e (c) qual K você escolheria.</span></div>

In [ ]:
# ✏️ Complete o código para testar diferentes valores de K

Ks      = range(1, 31)
acuracias_treino = []
acuracias_teste  = []

for k in Ks:
    # ✏️ Crie um KNeighborsClassifier com n_neighbors=k
    # ✏️ Treine com X_treino_sc e y_treino
    # ✏️ Calcule acurácia no treino e no teste
    # ✏️ Adicione às listas

    # knn_k = ???
    # knn_k.fit(???, ???)
    # acc_treino = ???
    # acc_teste  = ???
    # acuracias_treino.append(acc_treino)
    # acuracias_teste.append(acc_teste)
    pass

# ✏️ Plote os resultados:
# plt.plot(Ks, [a*100 for a in acuracias_treino], label="Treino")
# plt.plot(Ks, [a*100 for a in acuracias_teste],  label="Teste")
# plt.xlabel("K"); plt.ylabel("Acurácia (%)"); plt.legend()
# plt.show()

print("Complete o código acima!")


In [ ]:
# ── GABARITO DA MISSÃO 3 (descomente para ver) ───────────────────────────────
# Ks               = range(1, 31)
# acuracias_treino = []
# acuracias_teste  = []
#
# for k in Ks:
#     knn_k = KNeighborsClassifier(n_neighbors=k)
#     knn_k.fit(X_treino_sc, y_treino)
#     acuracias_treino.append(knn_k.score(X_treino_sc, y_treino))
#     acuracias_teste.append(knn_k.score(X_teste_sc,  y_teste))
#
# melhor_k    = Ks[np.argmax(acuracias_teste)]
# melhor_acc  = max(acuracias_teste)
#
# fig, ax = plt.subplots(figsize=(12, 5))
# ax.plot(Ks, [a*100 for a in acuracias_treino], "o--",
#         color="#e94560", linewidth=2, markersize=5, label="Treino")
# ax.plot(Ks, [a*100 for a in acuracias_teste],  "s-",
#         color="#0f3460", linewidth=2, markersize=5, label="Teste")
# ax.axvline(melhor_k, color="#f0a500", linestyle="--", linewidth=1.5,
#            label=f"Melhor K={melhor_k} ({melhor_acc:.1%})")
# ax.fill_betweenx([75, 100], 1, 4, alpha=0.07, color="#e94560",
#                  label="Zona de Overfitting")
# ax.fill_betweenx([75, 100], 22, 30, alpha=0.07, color="#0f3460",
#                  label="Zona de Underfitting")
# ax.set_xlabel("Valor de K"); ax.set_ylabel("Acurácia (%)")
# ax.set_title("Acurácia por K — Treino vs Teste", fontweight="bold")
# ax.legend(); ax.set_ylim(75, 100)
# plt.tight_layout(); plt.show()
#
# print(f"Melhor K: {melhor_k}  →  Acurácia no Teste: {melhor_acc:.4f} ({melhor_acc:.1%})")
# print(f"Regra √n sugeria: {int(np.sqrt(len(X_treino_sc)))}")


In [ ]:
# Executando o gabarito para continuar (mesmo que você não tenha completado)
Ks               = range(1, 31)
acuracias_treino = []
acuracias_teste  = []

for k in Ks:
    knn_k = KNeighborsClassifier(n_neighbors=k)
    knn_k.fit(X_treino_sc, y_treino)
    acuracias_treino.append(knn_k.score(X_treino_sc, y_treino))
    acuracias_teste.append(knn_k.score(X_teste_sc,   y_teste))

melhor_k   = list(Ks)[np.argmax(acuracias_teste)]
melhor_acc = max(acuracias_teste)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(Ks, [a*100 for a in acuracias_treino], "o--",
        color="#e94560", linewidth=2, markersize=5, label="Treino")
ax.plot(Ks, [a*100 for a in acuracias_teste],  "s-",
        color="#0f3460", linewidth=2, markersize=5, label="Teste")
ax.axvline(melhor_k, color="#f0a500", linestyle="--", linewidth=1.8,
           label=f"Melhor K = {melhor_k}  ({melhor_acc:.1%})")
ax.fill_betweenx([75, 100], 1, 3.5, alpha=0.06, color="#e94560")
ax.fill_betweenx([75, 100], 24, 30, alpha=0.06, color="#0f3460")
ax.annotate("Overfitting
(K pequeno)", xy=(2, 76.5), fontsize=9,
            color="#e94560", ha="center")
ax.annotate("Underfitting
(K grande)", xy=(27, 76.5), fontsize=9,
            color="#0f3460", ha="center")
ax.set_xlabel("Valor de K"); ax.set_ylabel("Acurácia (%)")
ax.set_title("Acurácia por Valor de K — Treino vs Teste  (Titanic, KNN)",
             fontweight="bold")
ax.legend(); ax.set_ylim(75, 100)
plt.tight_layout()
plt.savefig("aula05_curva_k.png", dpi=110, bbox_inches="tight")
plt.show()

print(f"✅  Melhor K encontrado: {melhor_k}")
print(f"    Acurácia no teste:   {melhor_acc:.4f}  ({melhor_acc:.1%})")
print(f"    Regra √n sugeria K = {int(np.sqrt(len(X_treino_sc)))}")


---

<div style="background: linear-gradient(90deg, #0f3460 0%, #16213e 100%); padding:20px 25px; border-radius:10px; margin:20px 0;"><div style="display:flex; justify-content:space-between; align-items:center; flex-wrap:wrap; gap:10px;"><div><span style="color:#e2b96f; font-size:0.85em; font-weight:bold; letter-spacing:2px;">PARTE 6</span><h2 style="color:white; margin:4px 0 0 0; font-family:'Segoe UI',sans-serif;">Avaliando o Modelo — Além da Acurácia</h2><p style="color:#a8d8ea; margin:6px 0 0 0; font-size:0.9em;">"Acurácia sozinha não conta a história completa."</p></div></div></div>


### Por que precisamos de mais de uma métrica?

Imagine que 62% dos passageiros do Titanic não sobreviveram.
Um modelo que **sempre prevê "não sobreviveu"** teria 62% de acurácia — sem aprender nada!

Por isso precisamos de métricas que olhem para cada classe separadamente:

| Métrica | O que mede | Quando priorizar |
|---------|-----------|-----------------|
| **Acurácia** | % total de acertos | Classes balanceadas |
| **Precisão** | Dos que eu disse "Sim", quantos eram reais? | Quando falso positivo é caro |
| **Recall** | Dos que eram reais "Sim", quantos achei? | Quando falso negativo é grave |
| **F1-Score** | Equilíbrio entre Precisão e Recall | Desbalanceamento |


In [ ]:
from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score
)

# Modelo com o melhor K encontrado
knn_final = KNeighborsClassifier(n_neighbors=melhor_k)
knn_final.fit(X_treino_sc, y_treino)
y_pred_final = knn_final.predict(X_teste_sc)

print(f"AVALIAÇÃO COMPLETA — KNN com K={melhor_k}")
print("=" * 50)
print(f"  Acurácia:  {accuracy_score(y_teste, y_pred_final):.4f}  ({accuracy_score(y_teste, y_pred_final):.1%})")
print(f"  Precisão:  {precision_score(y_teste, y_pred_final):.4f}")
print(f"  Recall:    {recall_score(y_teste, y_pred_final):.4f}")
print(f"  F1-Score:  {f1_score(y_teste, y_pred_final):.4f}")
print()
print("Relatório completo por classe:")
print(classification_report(y_teste, y_pred_final,
                             target_names=["Não Sobreviveu", "Sobreviveu"]))


In [ ]:
# Matriz de confusão — visualizando os erros em detalhe
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"Avaliação do KNN (K={melhor_k}) — Titanic",
             fontsize=13, fontweight="bold")

# Matriz de confusão
cm = confusion_matrix(y_teste, y_pred_final)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Não Sobrev.", "Sobreviveu"])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title("Matriz de Confusão", fontweight="bold")

# Anotações explicativas
vp = cm[1,1]; vn = cm[0,0]; fp = cm[0,1]; fn = cm[1,0]
axes[0].text(1.05, 0.5,
    f"VP={vp}: Sobreviventes
corretamente identificados

"
    f"VN={vn}: Não-sobreviventes
corretamente identificados

"
    f"FP={fp}: Disse 'Sobreviveu'
mas não sobreviveu

"
    f"FN={fn}: Disse 'Não Sobreviveu'
mas sobreviveu",
    transform=axes[0].transAxes, fontsize=8.5,
    verticalalignment="center", color="#333",
    bbox=dict(boxstyle="round", facecolor="#f8f9fa", alpha=0.8))

# Comparando métricas por classe
metricas_nome  = ["Acurácia", "Precisão
(Sobrev.)", "Recall
(Sobrev.)", "F1
(Sobrev.)"]
metricas_valor = [
    accuracy_score(y_teste, y_pred_final),
    precision_score(y_teste, y_pred_final),
    recall_score(y_teste, y_pred_final),
    f1_score(y_teste, y_pred_final),
]
cores_m = ["#a8d8ea","#0f3460","#e94560","#f0a500"]

barras = axes[1].bar(metricas_nome, [v*100 for v in metricas_valor],
                      color=cores_m, edgecolor="white", width=0.55)
for b, v in zip(barras, metricas_valor):
    axes[1].text(b.get_x() + b.get_width()/2, v*100 + 0.5,
                 f"{v:.1%}", ha="center", fontsize=11, fontweight="bold")
axes[1].set_ylabel("Valor (%)"); axes[1].set_ylim(0, 105)
axes[1].set_title("Métricas de Avaliação", fontweight="bold")

plt.tight_layout()
plt.savefig("aula05_avaliacao_final.png", dpi=110, bbox_inches="tight")
plt.show()


<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">MISSÃO 4 — Interprete os resultados olhando para a Matriz de Confusão e responda nas células abaixo: (a) quantos sobreviventes o modelo perdeu (não identificou)? (b) O Recall ou a Precisão está mais baixo? O que isso significa no contexto do Titanic? (c) Se você fosse um passageiro real, preferiria erro de FP ou FN?</span></div>

**✏️ Minhas respostas:**

**(a)** Sobreviventes não identificados (Falsos Negativos): `???`

**(b)** Recall / Precisão (escolha um): `???` está mais baixo — isso significa: `???`

**(c)** Como passageiro, prefiro erro de `???` porque: `???`


In [ ]:
# ── GABARITO DA MISSÃO 4 (descomente para ver) ───────────────────────────────
# cm = confusion_matrix(y_teste, y_pred_final)
# vp, fn, fp, vn = cm[1,1], cm[1,0], cm[0,1], cm[0,0]
#
# print(f"(a) Falsos Negativos (sobreviventes não identificados): {fn}")
# print(f"    O modelo disse 'não sobreviveu' para {fn} pessoas que sobreviveram de verdade.")
# print()
# prec = precision_score(y_teste, y_pred_final)
# rec  = recall_score(y_teste, y_pred_final)
# print(f"(b) Precisão = {prec:.1%}  |  Recall = {rec:.1%}")
# if rec < prec:
#     print("    Recall está mais baixo: o modelo perde alguns sobreviventes reais.")
#     print("    Prefere errar dizendo 'não sobreviveu' quando a pessoa sobreviveu (conservador).")
# else:
#     print("    Precisão está mais baixa: o modelo prevê sobrevivência para alguns que não sobrev.")
# print()
# print("(c) Como passageiro, é pior ser FN (modelo diz que você vai morrer quando sobreviverá).")
# print("    Na prática médica (ex: diagnóstico de câncer), FN é quase sempre mais grave que FP.")
# print("    Neste caso, preferiria um modelo com Recall maior, mesmo que Precisão caia um pouco.")


In [ ]:
# Comparação final: KNN sem norm vs com norm vs melhor K
print("RESUMO FINAL — Evolução do Modelo")
print("=" * 55)
print(f"  {'Configuração':<30} {'Acurácia':>10} {'F1-Score':>10}")
print("  " + "-"*53)

configs = [
    ("KNN K=5, sem normalização",   X_treino_raw, X_teste_raw,  5),
    ("KNN K=5, com normalização",   X_treino_sc,  X_teste_sc,   5),
    (f"KNN K={melhor_k}, com norm. (melhor)", X_treino_sc, X_teste_sc, melhor_k),
]
for nome, Xtr, Xte, k in configs:
    m = KNeighborsClassifier(n_neighbors=k).fit(Xtr, y_treino)
    yp = m.predict(Xte)
    acc = accuracy_score(y_teste, yp)
    f1  = f1_score(y_teste, yp)
    print(f"  {nome:<30} {acc:>10.1%} {f1:>10.4f}")

print()
print("Cada linha representa uma melhoria específica:")
print("  Linha 1→2: ganho da normalização")
print(f"  Linha 2→3: ganho do ajuste de K (5 → {melhor_k})")


---

## Checklist — O que você sabe fazer agora

| Habilidade | Praticada hoje? |
|------------|----------------|
| Explicar a intuição do KNN com votação de vizinhos | ☐ |
| Calcular distância Euclidiana manualmente | ☐ |
| Entender por que normalização é obrigatória para KNN | ☐ |
| Usar `KNeighborsClassifier.fit()` e `.predict()` | ☐ |
| Interpretar `classification_report` | ☐ |
| Ler uma Matriz de Confusão (VP, VN, FP, FN) | ☐ |
| Escolher o K ideal com curva treino/teste | ☐ |
| Diferenciar Precisão de Recall e saber quando cada importa | ☐ |

---

## Reflexão final

<div style="background:#fde8d8; border-left:5px solid #a04000; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#a04000;">🎯 </strong><span style="color:#a04000;">Escreva em suas próprias palavras: (1) o que é o KNN em uma frase simples, (2) qual foi o insight mais importante desta aula para você, (3) em que situação real você usaria KNN?</span></div>

---


**✏️ Minha reflexão:**

1. KNN em uma frase: *...*

2. Insight mais importante: *...*

3. Usaria KNN para: *...*


---

## O que vem a seguir?

O KNN é o ponto de partida. Nas próximas aulas você vai conhecer algoritmos mais poderosos —
e poderá comparar todos com o KNN como baseline de referência.

```
Já treinado:  KNN  ✅
Próximos:     Regressão Logística → SVM → Árvores de Decisão → Ensemble
```

<div style="background:#d4edda; border-left:5px solid #155724; padding:14px 20px; border-radius:6px; margin:12px 0;"><strong style="color:#155724;">✅ </strong><span style="color:#155724;">Parabéns! Você treinou seu primeiro modelo de Machine Learning. O <code>modelo.fit()</code> que você escreveu hoje usa a mesma interface de todos os outros algoritmos do scikit-learn — a partir de agora, aprender um novo algoritmo é entender a teoria e trocar o nome da classe.</span></div>

---

## Referências

- Scikit-Learn KNN: https://scikit-learn.org/stable/modules/neighbors.html
- Cover, T. & Hart, P. (1967). *Nearest neighbor pattern classification*. IEEE Transactions on Information Theory.
- Géron, A. (2019). *Hands-on Machine Learning*, Cap. 3. O'Reilly.
